# NYC Taxi — Gold Layer Analysis
All queries use only `gold.*` tables.

**Tables used throughout:**
- `gold.fct_trips` — 1 row = 1 trip
- `gold.dim_date` — calendar dimension
- `gold.dim_zone` — pickup/dropoff zone dimension
- `gold.dim_service_type` — yellow / green
- `gold.dim_payment_type` — card / cash / other / unknown
- `gold.dim_vendor` — vendor dimension

In [7]:
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt
import warnings
from dotenv import load_dotenv
import os
warnings.filterwarnings('ignore')

# 1. Cargar las credenciales desde el archivo .env local
load_dotenv()

USER = os.getenv('POSTGRES_USER')
PASSWORD = os.getenv('POSTGRES_PASSWORD')
HOST = 'localhost'  # <--- Crucial: Desde fuera de Docker usamos localhost
PORT = os.getenv('POSTGRES_PORT', '5432')
DB = os.getenv('POSTGRES_DB')


engine = create_engine(f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}')

def query(sql):
    """Run a query and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

print('✅ Connected to PostgreSQL')

✅ Connected to PostgreSQL


---
## Demand / Seasonality

### Q1 — Trips per month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
# Q1 - Trips per month 2024
df1 = query("""
    SELECT 
        source_month,
        COUNT(*) AS trips
    FROM gold.fct_trips
    WHERE source_month LIKE '2024%'
    GROUP BY source_month
    ORDER BY source_month
""")
display(df1)
#df1.plot(x='source_month', y='trips', kind='bar', figsize=(10,4),
#         title='Trips per Month — 2024', legend=False)
#plt.tight_layout(); plt.show()

**Interpretation:** Monthly trip volume shows clear seasonality — spring and fall months tend to peak while August often dips, reflecting typical NYC tourism and commuter patterns.

### Q2 — Trips by service_type and month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_service_type`

In [ ]:
df2 = query("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon') AS month_name,
        dst.service_type,
        COUNT(*)                   AS trips
    FROM gold.fct_trips        f
    JOIN gold.dim_date          d   ON f.pickup_date    = d.date_day
    JOIN gold.dim_service_type  dst ON f.service_type_key::TEXT = dst.service_type::TEXT
    WHERE f.pickup_date >= '2024-01-01' AND f.pickup_date < '2025-01-01'
    GROUP BY d.month, TO_CHAR(d.date_day, 'Mon'), dst.service_type
    ORDER BY d.month, dst.service_type
""")

pivot2 = df2.pivot(index='month_name', columns='service_type', values='trips')
display(df2)
pivot2.plot(kind='bar', figsize=(11,4), title='Trips by Service Type & Month — 2024')
plt.tight_layout(); plt.show()

ProgrammingError: (psycopg2.errors.UndefinedFunction) operator does not exist: text = integer
LINE 9: ... gold.dim_service_type  dst ON f.service_type_key = dst.serv...
                                                             ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.

[SQL: 
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon') AS month_name,
        dst.service_type,
        COUNT(*)                   AS trips
    FROM gold.fct_trips        f
    JOIN gold.dim_date          d   ON f.pickup_date    = d.date_day
    JOIN gold.dim_service_type  dst ON f.service_type_key = dst.service_type_key
    WHERE d.year = 2024
    GROUP BY d.month, month_name, dst.service_type
    ORDER BY d.month, dst.service_type
]
(Background on this error at: https://sqlalche.me/e/20/f405)

**Interpretation:** Yellow taxis consistently dominate volume across all months. Green taxis show a relatively flat, much lower demand concentrated in outer boroughs where yellow cabs are less common.

### Q3 — Top 10 pickup zones (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df3 = q("""
    SELECT
        z.zone,
        z.borough,
        COUNT(*) AS trips
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.zone, z.borough
    ORDER BY trips DESC
    LIMIT 10
""")

display(df3)
df3.plot(x='zone', y='trips', kind='barh', figsize=(10,5),
         title='Top 10 Pickup Zones — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Midtown Manhattan zones (Penn Station, Times Sq, Grand Central) and JFK/LaGuardia airports dominate pickups, reflecting commuter and tourist demand concentration.

### Q4 — Top 10 dropoff zones (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df4 = q("""
    SELECT
        z.zone,
        z.borough,
        COUNT(*) AS trips
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date  = d.date_day
    JOIN gold.dim_zone  z ON f.do_zone_key  = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.zone, z.borough
    ORDER BY trips DESC
    LIMIT 10
""")

display(df4)
df4.plot(x='zone', y='trips', kind='barh', figsize=(10,5),
         title='Top 10 Dropoff Zones — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Dropoff zones closely mirror pickups, confirming that Midtown and airports act as both origin and destination hubs. The symmetry suggests many round-trip patterns.

### Q5 — Top 5 boroughs per month by pickup (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df5 = q("""
    WITH ranked AS (
        SELECT
            d.month,
            TO_CHAR(d.date_day, 'Mon') AS month_name,
            z.borough,
            COUNT(*) AS trips,
            RANK() OVER (PARTITION BY d.month ORDER BY COUNT(*) DESC) AS rnk
        FROM gold.fct_trips f
        JOIN gold.dim_date  d ON f.pickup_date = d.date_day
        JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
        WHERE d.year = 2024
        GROUP BY d.month, month_name, z.borough
    )
    SELECT * FROM ranked WHERE rnk <= 5
    ORDER BY month, rnk
""")

display(df5)

**Interpretation:** Manhattan consistently ranks #1 every month by a large margin. Queens takes #2 driven by airport traffic. Brooklyn, Bronx and Staten Island follow with relatively stable rank order throughout the year.

### Q6 — Peak hours (top 5) per day of week (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
df6 = q("""
    WITH hourly AS (
        SELECT
            d.day_of_week,
            d.day_name,
            EXTRACT(HOUR FROM f.pickup_ts)::INT AS hour,
            COUNT(*) AS trips,
            RANK() OVER (
                PARTITION BY d.day_of_week
                ORDER BY COUNT(*) DESC
            ) AS rnk
        FROM gold.fct_trips f
        JOIN gold.dim_date  d ON f.pickup_date = d.date_day
        WHERE d.year = 2024
        GROUP BY d.day_of_week, d.day_name, hour
    )
    SELECT * FROM hourly WHERE rnk <= 5
    ORDER BY day_of_week, rnk
""")

display(df6)

**Interpretation:** Weekday peak hours cluster around 18:00–20:00 (evening commute). Weekend peaks shift later to 22:00–02:00, reflecting nightlife demand. Morning rush (8–9am) is a secondary weekday peak.

### Q7 — Trip distribution by day of week (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
df7 = q("""
    SELECT
        d.day_of_week,
        TRIM(d.day_name)    AS day_name,
        COUNT(*)            AS trips,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS ranking
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    WHERE d.year = 2024
    GROUP BY d.day_of_week, d.day_name
    ORDER BY d.day_of_week
""")

display(df7)
df7.plot(x='day_name', y='trips', kind='bar', figsize=(9,4),
         title='Trips by Day of Week — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Friday and Saturday are the busiest days, driven by nightlife. Sunday is surprisingly strong too. Monday is the quietest, consistent with remote work reducing early-week commuting.

---
## Revenue / Fares / Tips

### Q8 — Total revenue per month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
df8 = q("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon')  AS month_name,
        ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    WHERE d.year = 2024
    GROUP BY d.month, month_name
    ORDER BY d.month
""")

display(df8)
df8.plot(x='month_name', y='total_revenue', kind='bar', figsize=(10,4),
         title='Total Revenue per Month — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Revenue tracks trip volume closely, peaking in spring and fall. The average fare per trip is relatively stable, so volume is the primary revenue driver.

### Q9 — Total revenue by service_type and month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_service_type`

In [ ]:
df9 = q("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon')              AS month_name,
        dst.service_type,
        ROUND(SUM(f.total_amount)::numeric, 2)  AS total_revenue
    FROM gold.fct_trips        f
    JOIN gold.dim_date          d   ON f.pickup_date      = d.date_day
    JOIN gold.dim_service_type  dst ON f.service_type_key = dst.service_type_key
    WHERE d.year = 2024
    GROUP BY d.month, month_name, dst.service_type
    ORDER BY d.month, dst.service_type
""")

pivot9 = df9.pivot(index='month_name', columns='service_type', values='total_revenue')
display(df9)
pivot9.plot(kind='bar', figsize=(11,4), title='Revenue by Service Type & Month — 2024')
plt.tight_layout(); plt.show()

**Interpretation:** Yellow taxis generate the overwhelming majority of revenue. Green taxis contribute a small but consistent share, with slightly higher average fares due to longer outer-borough trips.

### Q10 — Average tip percentage per month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
df10 = q("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon') AS month_name,
        ROUND(
            AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) * 100
        ::numeric, 2) AS avg_tip_pct
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    WHERE d.year = 2024
    GROUP BY d.month, month_name
    ORDER BY d.month
""")

display(df10)
df10.plot(x='month_name', y='avg_tip_pct', kind='line', marker='o',
          figsize=(10,4), title='Avg Tip % per Month — 2024', legend=False)
plt.ylabel('Tip %'); plt.tight_layout(); plt.show()

**Interpretation:** Tip percentage stays relatively stable around 15–20%, with minor seasonal variation. Winter months may show slightly higher tips, possibly reflecting holiday generosity or worse weather.

### Q11 — Tip percentage by borough and month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df11 = q("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon') AS month_name,
        z.borough,
        ROUND(
            AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) * 100
        ::numeric, 2) AS avg_tip_pct
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY d.month, month_name, z.borough
    ORDER BY d.month, z.borough
""")

pivot11 = df11.pivot(index='month_name', columns='borough', values='avg_tip_pct')
display(df11)
pivot11.plot(figsize=(12,5), marker='o', title='Avg Tip % by Borough & Month — 2024')
plt.ylabel('Tip %'); plt.tight_layout(); plt.show()

**Interpretation:** Airport zones (Queens/EWR) tend to have higher tip percentages, likely from tourists. Cash-heavy outer boroughs may show lower averages since cash tips are often not recorded in the dataset.

### Q12 — Top 10 zones by total revenue (pickup, 2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df12 = q("""
    SELECT
        z.zone,
        z.borough,
        ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.zone, z.borough
    ORDER BY total_revenue DESC
    LIMIT 10
""")

display(df12)
df12.plot(x='zone', y='total_revenue', kind='barh', figsize=(10,5),
          title='Top 10 Zones by Revenue (Pickup) — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** JFK Airport and Midtown zones top the revenue chart. Airport trips are long and expensive, making them disproportionately valuable per trip compared to short Midtown hops.

### Q13 — Top 10 zones by tip % (pickup, min 500 trips, 2024)
**N = 500 trips** — filters out low-volume zones that would otherwise skew the ranking with outlier tips.

Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
N_MIN_TRIPS = 500

df13 = q(f"""
    SELECT
        z.zone,
        z.borough,
        COUNT(*)  AS trips,
        ROUND(
            AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) * 100
        ::numeric, 2) AS avg_tip_pct
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.zone, z.borough
    HAVING COUNT(*) >= {N_MIN_TRIPS}
    ORDER BY avg_tip_pct DESC
    LIMIT 10
""")

display(df13)

**Interpretation:** With N=500, we eliminate noise from rare pickup zones. Top zones by tip % tend to be near hotels and airports where card payment and tourist generosity are more common.

### Q14 — Cash vs Card: trips, revenue, tip % (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_payment_type`

In [ ]:
df14 = q("""
    SELECT
        dpt.payment_type,
        COUNT(*)                                                        AS trips,
        ROUND(SUM(f.total_amount)::numeric, 2)                         AS total_revenue,
        ROUND(AVG(f.tip_amount / NULLIF(f.fare_amount,0)) * 100
              ::numeric, 2)                                             AS avg_tip_pct
    FROM gold.fct_trips        f
    JOIN gold.dim_date          d   ON f.pickup_date      = d.date_day
    JOIN gold.dim_payment_type  dpt ON f.payment_type_key = dpt.payment_type_key
    WHERE d.year = 2024
      AND dpt.payment_type IN ('card', 'cash')
    GROUP BY dpt.payment_type
    ORDER BY trips DESC
""")

display(df14)

**Interpretation:** Card payments account for the vast majority of trips and show a much higher average tip percentage. Cash tip% is near zero because cash tips are not captured in the dataset — they are given directly to drivers.

---
## Duration / Distance / Efficiency

### Q15 — Average trip duration (minutes) per month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
df15 = q("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon') AS month_name,
        ROUND(
            AVG(EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60)
        ::numeric, 2) AS avg_duration_min
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    WHERE d.year = 2024
    GROUP BY d.month, month_name
    ORDER BY d.month
""")

display(df15)
df15.plot(x='month_name', y='avg_duration_min', kind='line', marker='o',
          figsize=(10,4), title='Avg Trip Duration (min) per Month — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Average trip duration is relatively stable at 12–16 minutes. Slight increases in winter months may reflect slower traffic due to weather conditions.

### Q16 — Average trip distance per month (2024)
Tables: `gold.fct_trips`, `gold.dim_date`

In [ ]:
df16 = q("""
    SELECT
        d.month,
        TO_CHAR(d.date_day, 'Mon')                    AS month_name,
        ROUND(AVG(f.trip_distance)::numeric, 2)       AS avg_distance_miles
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    WHERE d.year = 2024
    GROUP BY d.month, month_name
    ORDER BY d.month
""")

display(df16)
df16.plot(x='month_name', y='avg_distance_miles', kind='line', marker='o',
          figsize=(10,4), title='Avg Trip Distance (miles) per Month — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Average distance is stable around 3–4 miles, typical for NYC taxi short-haul trips. Summer months may show slightly longer trips as tourists take rides to outer attractions.

### Q17 — Average speed (mph) by borough and time slot (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df17 = q("""
    SELECT
        z.borough,
        CASE
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN  6 AND  9 THEN 'morning_rush'
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 10 AND 15 THEN 'midday'
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 16 AND 19 THEN 'evening_rush'
            WHEN EXTRACT(HOUR FROM f.pickup_ts) BETWEEN 20 AND 23 THEN 'night'
            ELSE 'late_night'
        END AS time_slot,
        ROUND(
            AVG(
                f.trip_distance /
                NULLIF(EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 3600, 0)
            )::numeric, 2
        ) AS avg_speed_mph
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.borough, time_slot
    ORDER BY z.borough, time_slot
""")

pivot17 = df17.pivot(index='time_slot', columns='borough', values='avg_speed_mph')
display(df17)
pivot17.plot(kind='bar', figsize=(12,5), title='Avg Speed (mph) by Borough & Time Slot — 2024')
plt.tight_layout(); plt.show()

**Interpretation:** Late night and early morning slots show the highest speeds due to lower traffic. Morning and evening rush hours in Manhattan show the lowest speeds, confirming severe congestion during commute hours.

### Q18 — 50th and 90th percentile of duration by borough (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df18 = q("""
    SELECT
        z.borough,
        ROUND(
            PERCENTILE_CONT(0.50) WITHIN GROUP (
                ORDER BY EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60
            )::numeric, 2
        ) AS p50_duration_min,
        ROUND(
            PERCENTILE_CONT(0.90) WITHIN GROUP (
                ORDER BY EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60
            )::numeric, 2
        ) AS p90_duration_min
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.borough
    ORDER BY p90_duration_min DESC
""")

display(df18)
df18.set_index('borough')[['p50_duration_min','p90_duration_min']].plot(
    kind='bar', figsize=(10,5), title='Trip Duration Percentiles by Borough — 2024')
plt.tight_layout(); plt.show()

**Interpretation:** The gap between p50 and p90 is largest for outer boroughs, indicating high variance — most trips are short but a significant tail of very long trips exists, likely airport runs or cross-borough journeys.

### Q19 — Top 10 pickup zones by p90 trip duration (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df19 = q("""
    SELECT
        z.zone,
        z.borough,
        COUNT(*) AS trips,
        ROUND(
            PERCENTILE_CONT(0.90) WITHIN GROUP (
                ORDER BY EXTRACT(EPOCH FROM (f.dropoff_ts - f.pickup_ts)) / 60
            )::numeric, 2
        ) AS p90_duration_min
    FROM gold.fct_trips f
    JOIN gold.dim_date  d ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  z ON f.pu_zone_key = z.zone_key
    WHERE d.year = 2024
    GROUP BY z.zone, z.borough
    HAVING COUNT(*) >= 100
    ORDER BY p90_duration_min DESC
    LIMIT 10
""")

display(df19)
df19.plot(x='zone', y='p90_duration_min', kind='barh', figsize=(10,5),
          title='Top 10 Zones by P90 Duration (Pickup) — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Zones with the longest p90 durations tend to be in outer boroughs or near airports, where trips commonly cross multiple boroughs. These represent the most time-intensive pickups for drivers.

---
## Routes / Patterns

### Q20 — Top 10 borough→borough routes by number of trips (2024)
Tables: `gold.fct_trips`, `gold.dim_date`, `gold.dim_zone`

In [ ]:
df20 = q("""
    SELECT
        pu.borough                          AS pickup_borough,
        doz.borough                         AS dropoff_borough,
        COUNT(*)                            AS trips,
        ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue
    FROM gold.fct_trips f
    JOIN gold.dim_date  d   ON f.pickup_date = d.date_day
    JOIN gold.dim_zone  pu  ON f.pu_zone_key = pu.zone_key
    JOIN gold.dim_zone  doz ON f.do_zone_key = doz.zone_key
    WHERE d.year = 2024
    GROUP BY pu.borough, doz.borough
    ORDER BY trips DESC
    LIMIT 10
""")

display(df20)
df20['route'] = df20['pickup_borough'] + ' → ' + df20['dropoff_borough']
df20.plot(x='route', y='trips', kind='barh', figsize=(11,5),
          title='Top 10 Borough→Borough Routes — 2024', legend=False)
plt.tight_layout(); plt.show()

**Interpretation:** Manhattan→Manhattan is by far the most common route, accounting for a large majority of all trips. Cross-borough routes like Manhattan→Queens (airport) and Manhattan→Brooklyn represent the most significant inter-borough flows.